**SVM Asset Allocation**

**1. Data**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

folder = Path(".")
data = pd.read_csv(
    folder / "monthly_returns.csv", index_col="Date", parse_dates=True
)

**2. Preprocessing**

In [ ]:
data = data.apply(pd.to_numeric, errors="coerce")
data = data.replace([np.inf, -np.inf], np.nan).dropna().sort_index()
benchmark = data.pop("all_weather_return")
assert data.index.is_unique and np.isfinite(data).all().all()

**3. SVM Signals**

The probabilities below were generated by our trained SVM models; training is omitted here for brevity.

In [ ]:
probability = pd.read_csv(
    folder / "svm_probabilities.csv", index_col="Date", parse_dates=True
).dropna()
returns = data[probability.columns]

**4. Portfolio Performance**

In [ ]:
def svm_weights(returns, probability):
    volatility = returns.rolling(12).std().shift(1) * np.sqrt(12)
    volatility = volatility.loc[probability.index].clip(lower=0.001)
    inverse = probability.gt(0.5) / volatility
    return inverse.div(inverse.sum(axis=1), axis=0).fillna(0)

weights = svm_weights(returns, probability)
svm_return = (weights * returns.loc[weights.index]).sum(axis=1)
result = pd.DataFrame({"All Weather": benchmark.loc[weights.index],
                       "SVM Screening": svm_return})
growth = (1 + result).cumprod()
ax = growth.plot(figsize=(7.2, 3), color=["blue", "red"], linewidth=1.8)
ax.set(
    xlabel="Year", ylabel="Cumulative wealth", title="Portfolio Performance"
)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()